## **Datenvorbereitung**

In diesem Notebook wird der im Rahmen der explorativen Datenanalyse erstellte Datensatz für die anschließende Modellierung vorbereitet. Ziel ist es, eine konsistente und vollständige Datenbasis zu erzeugen, ohne relevante Beobachtungen oder Rezessionsereignisse unnötig zu verlieren. Ein besonderer Schwerpunkt liegt zunächst auf der Analyse und Behandlung fehlender Werte.

**Laden des EDA-Datensatzes**

Als Ausgangspunkt wird der zuvor erstellte EDA-Datensatz geladen. Dieser enthält die monatlichen makroökonomischen Prädiktorvariablen sowie die binäre Zielvariable Rezession.

Nach dem Import werden Dimension, Spaltenstruktur und erste Beobachtungen kontrolliert, um sicherzustellen, dass der Datensatz korrekt eingelesen wurde.

In [35]:
import pandas as pd

# Load the Excel file
try:
    df = pd.read_excel('eda_dataset.xlsx')
except FileNotFoundError:
    print("Fehler: 'eda_dataset.xlsx' wurde nicht gefunden. Bitte stellen Sie sicher, dass die Datei im aktuellen Verzeichnis liegt.")
    df = None # Set df to None to prevent further errors

if df is not None:
    print("\n--- Grundkontrolle des Datensatzes ---")

    # 1. Dimensionen
    print(f"\nDimensionen des Datensatzes (Zeilen, Spalten): {df.shape}")

    # 2. Spaltennamen
    print("\nSpaltennamen:")
    for col in df.columns:
        print(f"- {col}")

    # 3. Datentypen und Nicht-Null-Werte
    print("\nDatentypen und Nicht-Null-Werte pro Spalte:")
    df.info()

    # 4. Zeitraum (Annahme: 'Datum' Spalte existiert)
    if 'Datum' in df.columns:
        try:
            df['Datum'] = pd.to_datetime(df['Datum'])
            print(f"\nZeitraum des Datensatzes: von {df['Datum'].min().strftime('%Y-%m-%d')} bis {df['Datum'].max().strftime('%Y-%m-%d')}")
        except Exception as e:
            print(f"\nWarnung: 'Datum' Spalte konnte nicht in Datetime konvertiert werden. Fehler: {e}")
            print(f"Erste und letzte Werte der 'Datum' Spalte: {df['Datum'].min()} bis {df['Datum'].max()}")
    else:
        print("\nWarnung: 'Datum' Spalte nicht gefunden, Zeitraum kann nicht bestimmt werden.")

    # 5. Zielvariable (Annahme: 'Rezession' als Zielvariable)
    target_variable = 'Rezession'
    if target_variable in df.columns:
        print(f"\nInformationen zur Zielvariable '{target_variable}':")
        print(f"Datentyp: {df[target_variable].dtype}")
        print(f"Wertverteilung:\n{df[target_variable].value_counts()}")
    else:
        print(f"\nWarnung: Zielvariable '{target_variable}' nicht gefunden. Bitte überprüfen Sie den Spaltennamen.")


--- Grundkontrolle des Datensatzes ---

Dimensionen des Datensatzes (Zeilen, Spalten): (425, 30)

Spaltennamen:
- Datum
- Industrial_Production
- Manufacturing_Output
- Capital_Goods_Output
- Intermediate_Goods_Output
- Consumer_Goods_Output
- Orders_Abroad_Intermediate_Capital
- Construction_Orders
- Unemployment_Rate
- Employment
- CPI
- Inflation_Rate
- German_Bond_Yield
- Effective_Exchange_Rate
- Consumer_Confidence
- M1_Index
- M3_Index
- Orders_Inflow
- Exchange_Rate_USD_EUR
- Domestic_Capital_Goods_Orders
- Domestic_Intermediate_Goods_Orders
- Euribor_3M
- Brent_Oil
- Production_Expectations_Manufacturing
- Business_Situation_Services
- Business_Situation_Retail
- Term_Spread_Germany
- Exports
- Imports
- Rezession

Datentypen und Nicht-Null-Werte pro Spalte:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 425 entries, 0 to 424
Data columns (total 30 columns):
 #   Column                                 Non-Null Count  Dtype         
---  ------                              

**- Behandlung fehlender Werte**

Zunächst wird systematisch untersucht, welche Variablen fehlende Beobachtungen enthalten und an welcher zeitlichen Position diese auftreten.

Die Analyse ergab insgesamt 103 fehlende Werte. Die Zielvariable Rezession enthält keine fehlenden Werte.

Besonders auffällig waren drei Variablen mit längeren strukturellen Randlücken:

**Business_Situation_Services: 51 fehlende Werte am Anfang der Zeitreihe**

**Unemployment_Rate: 11 fehlende Werte am Anfang der Zeitreihe**

**Term_Spread_Germany: 29 fehlende Werte am Ende der Zeitreihe**

Daneben wiesen mehrere Variablen lediglich eine einzelne fehlende Beobachtung am aktuellen Rand des Datensatzes auf.


Im ersten Schritt der Behandlung fehlender Werte werden ausschließlich interne Lücken innerhalb der Zeitreihen mittels linearer Interpolation geschlossen. Zuvor wird die Variable Datum in ein Datumsformat konvertiert und der Datensatz chronologisch sortiert. Für die Interpolation werden ausschließlich numerische Prädiktorvariablen berücksichtigt. Die Zielvariable Rezession wird ausdrücklich ausgeschlossen und nicht verändert. Durch die Einstellung limit_area='inside' werden nur fehlende Werte interpoliert, die zwischen zwei vorhandenen Beobachtungen liegen. Fehlende Werte am Anfang oder Ende einer Zeitreihe bleiben zunächst bestehen und werden anschließend separat behandelt.

In [36]:
import pandas as pd
import numpy as np

# 1. Datum vorbereiten und chronologisch sortieren
df["Datum"] = pd.to_datetime(df["Datum"])
df = df.sort_values("Datum").reset_index(drop=True)

# 2. Numerische Prädiktorvariablen auswählen
# Datum und Zielvariable Rezession werden nicht interpoliert
feature_cols = df.select_dtypes(include=np.number).columns.drop(
    "Rezession", errors="ignore"
)

# 3. Ausschließlich interne Datenlücken linear interpolieren
df[feature_cols] = df[feature_cols].interpolate(
    method="linear",
    limit_area="inside"
)

# 4. Verbleibende fehlende Werte kontrollieren
missing_after = df.isna().sum()
missing_summary = missing_after[missing_after > 0]

print("--- Fehlende Werte nach der Interpolation ---")

if missing_summary.empty:
    print("Keine fehlenden Werte mehr vorhanden.")
else:
    print("Verbleibende Randlücken:")
    print(missing_summary)

# 5. Kontrolle der Zielvariable
print("\nFehlende Werte in Rezession:")
print(df["Rezession"].isna().sum())

--- Fehlende Werte nach der Interpolation ---
Verbleibende Randlücken:
Industrial_Production                  1
Manufacturing_Output                   1
Capital_Goods_Output                   1
Intermediate_Goods_Output              1
Consumer_Goods_Output                  1
Orders_Abroad_Intermediate_Capital     1
Construction_Orders                    1
Unemployment_Rate                     11
Orders_Inflow                          1
Domestic_Capital_Goods_Orders          1
Domestic_Intermediate_Goods_Orders     1
Business_Situation_Services           51
Term_Spread_Germany                   29
Exports                                1
Imports                                1
dtype: int64

Fehlende Werte in Rezession:
0


**Analyse der verbleibenden strukturellen Randlücken**

Nach der Behandlung interner Datenlücken mittels linearer Interpolation verbleiben strukturelle Randlücken am Anfang bzw. Ende einzelner Zeitreihen. Diese entstehen insbesondere durch unterschiedliche Verfügbarkeitszeiträume der verwendeten Indikatoren.

Da Rezessionsmonate im Datensatz eine Minderheitsklasse darstellen, soll auf eine pauschale Löschung unvollständiger Beobachtungen (dropna) verzichtet werden. Eine solche Vorgehensweise könnte insbesondere zum Verlust relevanter Rezessionsbeobachtungen führen.

Daher wird zunächst untersucht, wie viele Beobachtungen von fehlenden Werten betroffen sind, in welchem Zeitraum diese auftreten und wie viele Rezessionsmonate darunter liegen. Die Ergebnisse dienen anschließend als Grundlage für die gezielte Behandlung der verbleibenden Randlücken.

In [37]:
print("--- Auswirkungen der verbleibenden Randlücken ---")

# Zeilen mit mindestens einem fehlenden Feature
rows_with_nan = df[df.isna().any(axis=1)]

print("Gesamtzahl Beobachtungen:", len(df))
print("Beobachtungen mit mindestens einem NaN:", len(rows_with_nan))
print("Vollständige Beobachtungen:", len(df.dropna()))

print("\nZeitraum mit NaN:")
print(rows_with_nan["Datum"].min(), "bis", rows_with_nan["Datum"].max())

print("\nRezessionsmonate mit mindestens einem NaN:")
print(rows_with_nan["Rezession"].value_counts().sort_index())

print("\nAnzahl betroffener Rezessionsmonate:")
print(rows_with_nan["Rezession"].sum())

--- Auswirkungen der verbleibenden Randlücken ---
Gesamtzahl Beobachtungen: 425
Beobachtungen mit mindestens einem NaN: 80
Vollständige Beobachtungen: 345

Zeitraum mit NaN:
1991-01-31 00:00:00 bis 2026-05-31 00:00:00

Rezessionsmonate mit mindestens einem NaN:
Rezession
0    51
1    29
Name: count, dtype: int64

Anzahl betroffener Rezessionsmonate:
29


**Behandlung minimaler Randlücken mittels Forward Fill**

Nach der Analyse der verbleibenden Randlücken werden zunächst Variablen behandelt, bei denen lediglich ein einzelner Wert am Rand der Zeitreihe fehlt. Hierfür wird die Methode Forward Fill (ffill) verwendet. Dabei wird der zuletzt verfügbare Beobachtungswert für den unmittelbar folgenden fehlenden Zeitpunkt übernommen.

Die drei Variablen Business_Situation_Services, Unemployment_Rate und Term_Spread_Germany werden von diesem Schritt ausgeschlossen, da sie längere strukturelle Randlücken aufweisen und anschließend separat behandelt werden. Die Zielvariable Rezession sowie die Zeitvariable Datum bleiben ebenfalls unverändert.

In [38]:
print("--- Behandlung minimaler Randlücken ---")

# Große strukturelle Randlücken separat behandeln
large_gap_cols = [
    "Business_Situation_Services",
    "Unemployment_Rate",
    "Term_Spread_Germany"
]

# Nur Variablen mit genau einem fehlenden Wert behandeln
for col in df.columns:
    if col not in ["Datum", "Rezession"] + large_gap_cols:

        if df[col].isna().sum() == 1:
            df[col] = df[col].ffill()
            print(f"{col}: 1 Randlücke mit Forward Fill behandelt.")

# Kontrolle
print("\nVerbleibende fehlende Werte:")
missing = df.isna().sum()
print(missing[missing > 0])

print("\nAnzahl Beobachtungen:", len(df))
print("Rezessionsmonate:", df["Rezession"].sum())

--- Behandlung minimaler Randlücken ---
Industrial_Production: 1 Randlücke mit Forward Fill behandelt.
Manufacturing_Output: 1 Randlücke mit Forward Fill behandelt.
Capital_Goods_Output: 1 Randlücke mit Forward Fill behandelt.
Intermediate_Goods_Output: 1 Randlücke mit Forward Fill behandelt.
Consumer_Goods_Output: 1 Randlücke mit Forward Fill behandelt.
Orders_Abroad_Intermediate_Capital: 1 Randlücke mit Forward Fill behandelt.
Construction_Orders: 1 Randlücke mit Forward Fill behandelt.
Orders_Inflow: 1 Randlücke mit Forward Fill behandelt.
Domestic_Capital_Goods_Orders: 1 Randlücke mit Forward Fill behandelt.
Domestic_Intermediate_Goods_Orders: 1 Randlücke mit Forward Fill behandelt.
Exports: 1 Randlücke mit Forward Fill behandelt.
Imports: 1 Randlücke mit Forward Fill behandelt.

Verbleibende fehlende Werte:
Unemployment_Rate              11
Business_Situation_Services    51
Term_Spread_Germany            29
dtype: int64

Anzahl Beobachtungen: 425
Rezessionsmonate: 88


**Auswahl geeigneter Prädiktoren für die Regressions-Imputation**

Für die drei Variablen mit längeren strukturellen Randlücken (Unemployment_Rate, Business_Situation_Services und Term_Spread_Germany) sollen die fehlenden Werte anschließend modellbasiert geschätzt werden.

Zur Vorbereitung werden zunächst die linearen Korrelationen zwischen der jeweils unvollständigen Variable und den übrigen numerischen Prädiktoren untersucht. Die Zielvariable Rezession wird dabei ausdrücklich ausgeschlossen, damit sie nicht zur Imputation der Eingangsvariablen verwendet wird.

Für jede der drei Variablen werden die zehn betragsmäßig stärksten Korrelationen ausgegeben. Diese Analyse dient als erste Orientierung für die Auswahl geeigneter Prädiktoren für die anschließende Regressions-Imputation. Eine hohe Korrelation allein bedeutet dabei noch nicht automatisch, dass eine Variable als Prädiktor übernommen wird.

In [39]:
print("--- Auswahl geeigneter Prädiktoren für die Regressions-Imputation ---")

problem_cols = [
    "Unemployment_Rate",
    "Business_Situation_Services",
    "Term_Spread_Germany"
]

# Nur numerische Variablen
numeric_df = df.select_dtypes(include=np.number).copy()

# Zielvariable Rezession nicht als Prädiktor verwenden
numeric_df = numeric_df.drop(columns=["Rezession"], errors="ignore")

for target in problem_cols:

    correlations = (
        numeric_df.corr()[target]
        .drop(labels=problem_cols, errors="ignore")
        .dropna()
        .abs()
        .sort_values(ascending=False)
    )

    print(f"\n{target}")
    print("Stärkste Korrelationen:")
    print(correlations.head(10))

--- Auswahl geeigneter Prädiktoren für die Regressions-Imputation ---

Unemployment_Rate
Stärkste Korrelationen:
Employment                            0.884643
M1_Index                              0.824925
M3_Index                              0.819193
Exports                               0.802031
Imports                               0.787815
CPI                                   0.765451
Capital_Goods_Output                  0.742226
Business_Situation_Retail             0.733005
German_Bond_Yield                     0.732374
Orders_Abroad_Intermediate_Capital    0.729127
Name: Unemployment_Rate, dtype: float64

Business_Situation_Services
Stärkste Korrelationen:
M3_Index                              0.545586
CPI                                   0.532018
M1_Index                              0.507913
Employment                            0.477847
Construction_Orders                   0.439561
Orders_Abroad_Intermediate_Capital    0.405653
German_Bond_Yield                     0.36

**Detaillierte Prüfung potenzieller Prädiktoren**

Nach der Identifikation der betragsmäßig stärksten Korrelationen werden die potenziellen Prädiktoren detaillierter untersucht. Dabei wird das Vorzeichen des Korrelationskoeffizienten beibehalten, um neben der Stärke auch die Richtung des linearen Zusammenhangs zu erkennen.

Positive Werte weisen auf einen gleichgerichteten, negative Werte auf einen gegenläufigen Zusammenhang hin. Die Zielvariable Rezession sowie die jeweils anderen problematischen Variablen werden nicht als potenzielle Prädiktoren berücksichtigt.

Die Ergebnisse unterstützen die anschließende Auswahl eines geeigneten und möglichst sparsamen Sets von Prädiktoren für die Regressions-Imputation.

In [40]:
print("--- Detaillierte Prüfung potenzieller Prädiktoren ---")

problem_cols = [
    "Unemployment_Rate",
    "Business_Situation_Services",
    "Term_Spread_Germany"
]

numeric_df = df.select_dtypes(include=np.number).drop(
    columns=["Rezession"], errors="ignore"
)

for target in problem_cols:

    corr = (
        numeric_df.corr()[target]
        .drop(labels=problem_cols, errors="ignore")
        .dropna()
    )

    # nach absoluter Stärke sortieren,
    # aber ursprüngliches Vorzeichen behalten
    corr = corr.loc[corr.abs().sort_values(ascending=False).index]

    print(f"\n{target}")
    print(corr.head(10))

--- Detaillierte Prüfung potenzieller Prädiktoren ---

Unemployment_Rate
Employment                           -0.884643
M1_Index                             -0.824925
M3_Index                             -0.819193
Exports                              -0.802031
Imports                              -0.787815
CPI                                  -0.765451
Capital_Goods_Output                 -0.742226
Business_Situation_Retail            -0.733005
German_Bond_Yield                     0.732374
Orders_Abroad_Intermediate_Capital   -0.729127
Name: Unemployment_Rate, dtype: float64

Business_Situation_Services
M3_Index                             -0.545586
CPI                                  -0.532018
M1_Index                             -0.507913
Employment                           -0.477847
Construction_Orders                   0.439561
Orders_Abroad_Intermediate_Capital   -0.405653
German_Bond_Yield                     0.367358
Consumer_Confidence                   0.361908
Exports     

**Bestimmung der Prädiktoren für die modellbasierte Imputation**

Für die Vervollständigung der Variablen mit umfangreichen strukturellen Randlücken (Unemployment_Rate, Business_Situation_Services und Term_Spread_Germany) wird eine modellbasierte Regressions-Imputation eingesetzt.

Die Auswahl geeigneter Prädiktoren erfolgt in zwei Schritten. Zunächst werden anhand der zuvor berechneten Korrelationskoeffizienten Variablen mit vergleichsweise starken linearen Zusammenhängen zur jeweils zu imputierenden Variable als potenzielle Prädiktoren identifiziert. Anschließend wird aus diesem Kandidatenpool mittels schrittweiser Vorwärtsauswahl (Forward Selection) ein reduziertes Prädiktorenset bestimmt.

Beginnend mit dem Prädiktor mit der höchsten Erklärungskraft werden weitere Variablen sukzessive aufgenommen. Eine zusätzliche Variable wird nur berücksichtigt, wenn sich das Bestimmtheitsmaß R
2
 des Regressionsmodells um mehr als 0,01 verbessert. Dadurch wird ein möglichst kompaktes Imputationsmodell angestrebt und die Aufnahme von Variablen mit nur geringem zusätzlichen Erklärungsbeitrag begrenzt.

Die Zielvariable Rezession wird vollständig von der Prädiktorenauswahl und Imputation ausgeschlossen. Dadurch wird verhindert, dass Informationen aus der später vorherzusagenden Zielvariable zur Konstruktion der Eingangsvariablen verwendet werden.

Die ermittelten R
2
-Werte werden ausschließlich als In-Sample-Gütemaß der Imputationsmodelle interpretiert. Sie beschreiben somit die Anpassung der Regressionsmodelle an die beobachteten Werte und stellen keine Bewertung der späteren Out-of-Sample-Prognoseleistung des Rezessionsmodells dar.

Die ausgewählten Prädiktoren bilden anschließend die Grundlage für die Regressions-Imputation der verbleibenden strukturellen Randlücken.

In [41]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

print("--- Auswahl der Prädiktoren für die Regressions-Imputation ---")

candidate_sets = {

    "Unemployment_Rate": [
        "Employment",
        "M1_Index",
        "M3_Index",
        "Exports",
        "Imports",
        "CPI"
    ],

    "Business_Situation_Services": [
        "M3_Index",
        "CPI",
        "M1_Index",
        "Employment",
        "Construction_Orders",
        "Consumer_Confidence"
    ],

    "Term_Spread_Germany": [
        "Consumer_Goods_Output",
        "Inflation_Rate",
        "Euribor_3M",
        "Business_Situation_Retail",
        "Employment"
    ]
}

selected_predictors = {}

for target, candidates in candidate_sets.items():

    data = df[[target] + candidates].dropna()

    remaining = candidates.copy()
    selected = []
    best_r2 = -np.inf

    while remaining:

        results = []

        for candidate in remaining:

            predictors = selected + [candidate]

            X = data[predictors]
            y = data[target]

            model = LinearRegression()
            model.fit(X, y)

            pred = model.predict(X)
            r2 = r2_score(y, pred)

            results.append((candidate, r2))

        candidate, new_r2 = max(
            results,
            key=lambda x: x[1]
        )

        # Variable nur aufnehmen, wenn sie das Modell verbessert
        if new_r2 > best_r2 + 0.01:

            selected.append(candidate)
            remaining.remove(candidate)
            best_r2 = new_r2

        else:
            break

    selected_predictors[target] = selected

    print(f"\n{target}")
    print("Ausgewählte Prädiktoren:", selected)
    print(f"R² des Modells: {best_r2:.3f}")

--- Auswahl der Prädiktoren für die Regressions-Imputation ---

Unemployment_Rate
Ausgewählte Prädiktoren: ['Employment', 'CPI', 'Imports', 'M1_Index', 'M3_Index']
R² des Modells: 0.909

Business_Situation_Services
Ausgewählte Prädiktoren: ['M3_Index', 'Consumer_Confidence', 'Construction_Orders', 'Employment', 'CPI', 'M1_Index']
R² des Modells: 0.671

Term_Spread_Germany
Ausgewählte Prädiktoren: ['Consumer_Goods_Output', 'Euribor_3M', 'Employment', 'Inflation_Rate']
R² des Modells: 0.744


**Regressions-Imputation der strukturellen Randlücken**

Nach der Auswahl geeigneter Prädiktoren werden die verbleibenden strukturellen Randlücken mittels linearer Regressions-Imputation geschlossen. Für jede der drei betroffenen Variablen (Unemployment_Rate, Business_Situation_Services und Term_Spread_Germany) wird ein separates lineares Regressionsmodell mit den zuvor ausgewählten Prädiktoren erstellt.

Die Modelle werden ausschließlich auf Beobachtungen trainiert, für die sowohl die zu imputierende Variable als auch sämtliche benötigten Prädiktoren vollständig vorliegen. Anschließend werden die trainierten Modelle verwendet, um ausschließlich die fehlenden Werte der jeweiligen Variable zu schätzen. Bereits vorhandene Originalwerte werden nicht verändert.

In [42]:
from sklearn.linear_model import LinearRegression

print("--- Regressions-Imputation der strukturellen Randlücken ---")

selected_predictors = {
    "Unemployment_Rate": [
        "Employment", "CPI", "Imports", "M1_Index", "M3_Index"
    ],
    "Business_Situation_Services": [
        "M3_Index", "Consumer_Confidence", "Construction_Orders",
        "Employment", "CPI", "M1_Index"
    ],
    "Term_Spread_Germany": [
        "Consumer_Goods_Output", "Euribor_3M",
        "Employment", "Inflation_Rate"
    ]
}

for target, predictors in selected_predictors.items():

    # Nur vollständige Originalbeobachtungen zum Trainieren verwenden
    train_mask = df[target].notna() & df[predictors].notna().all(axis=1)

    # Nur fehlende Werte der Zielvariable imputieren
    missing_mask = df[target].isna() & df[predictors].notna().all(axis=1)

    X_train = df.loc[train_mask, predictors]
    y_train = df.loc[train_mask, target]

    model = LinearRegression()
    model.fit(X_train, y_train)

    # Fehlende Randwerte schätzen
    df.loc[missing_mask, target] = model.predict(
        df.loc[missing_mask, predictors]
    )

    print(f"{target}: {missing_mask.sum()} Werte imputiert.")

# Kontrolle
print("\n--- Kontrolle nach der Imputation ---")

missing = df.isna().sum()
print("Verbleibende fehlende Werte:")
print(missing[missing > 0])

print("\nAnzahl Beobachtungen:", len(df))
print("Rezessionsmonate:", df["Rezession"].sum())
print("Fehlende Werte in Rezession:", df["Rezession"].isna().sum())

--- Regressions-Imputation der strukturellen Randlücken ---
Unemployment_Rate: 11 Werte imputiert.
Business_Situation_Services: 51 Werte imputiert.
Term_Spread_Germany: 29 Werte imputiert.

--- Kontrolle nach der Imputation ---
Verbleibende fehlende Werte:
Series([], dtype: int64)

Anzahl Beobachtungen: 425
Rezessionsmonate: 88
Fehlende Werte in Rezession: 0


**Plausibilitätskontrolle der imputierten Randwerte**

Nach Abschluss der Regressions-Imputation werden die geschätzten Randwerte einer Plausibilitätskontrolle unterzogen. Ziel ist es zu überprüfen, ob durch die Imputation auffällige oder offensichtlich unrealistische Werte entstanden sind.

Hierfür werden für die drei imputierten Variablen Unemployment_Rate, Business_Situation_Services und Term_Spread_Germany zunächst Minimum, Maximum und Mittelwert des gesamten jeweiligen Merkmals betrachtet. Anschließend werden die konkret imputierten Zeiträume separat ausgegeben.

Dadurch können die geschätzten Werte hinsichtlich ihres Niveaus und zeitlichen Verlaufs nachvollzogen und mit dem Wertebereich der jeweiligen Gesamtzeitreihe verglichen werden. Dieser Schritt dient als zusätzliche Qualitätskontrolle der modellbasierten Imputation.

In [43]:
print("--- Plausibilitätskontrolle der imputierten Randwerte ---")

variables = [
    "Unemployment_Rate",
    "Business_Situation_Services",
    "Term_Spread_Germany"
]

for var in variables:
    print(f"\n{var}")
    print("-" * 40)

    print("Minimum:", round(df[var].min(), 3))
    print("Maximum:", round(df[var].max(), 3))
    print("Mittelwert:", round(df[var].mean(), 3))

# Spezifische imputierte Zeiträume anzeigen
print("\n--- Imputierte Werte: Unemployment_Rate ---")
print(
    df.loc[
        df["Datum"] <= "1991-11-30",
        ["Datum", "Unemployment_Rate"]
    ].to_string(index=False)
)

print("\n--- Imputierte Werte: Business_Situation_Services ---")
print(
    df.loc[
        df["Datum"] <= "1995-03-31",
        ["Datum", "Business_Situation_Services"]
    ].to_string(index=False)
)

print("\n--- Imputierte Werte: Term_Spread_Germany ---")
print(
    df.loc[
        df["Datum"] >= "2024-01-31",
        ["Datum", "Term_Spread_Germany"]
    ].to_string(index=False)
)

--- Plausibilitätskontrolle der imputierten Randwerte ---

Unemployment_Rate
----------------------------------------
Minimum: 4.9
Maximum: 12.1
Mittelwert: 8.034

Business_Situation_Services
----------------------------------------
Minimum: -53.4
Maximum: 59.5
Mittelwert: 13.925

Term_Spread_Germany
----------------------------------------
Minimum: -2.381
Maximum: 3.169
Mittelwert: 0.638

--- Imputierte Werte: Unemployment_Rate ---
     Datum  Unemployment_Rate
1991-01-31           6.398473
1991-02-28           6.758523
1991-03-31           6.895596
1991-04-30           7.052830
1991-05-31           6.828297
1991-06-30           7.207703
1991-07-31           7.441133
1991-08-31           7.747057
1991-09-30           7.849163
1991-10-31           8.100776
1991-11-30           8.066287

--- Imputierte Werte: Business_Situation_Services ---
     Datum  Business_Situation_Services
1991-01-31                    21.344079
1991-02-28                    21.636664
1991-03-31                  

**Fehlender Werte prüfen**

Nach Abschluss aller Imputationsschritte wird eine abschließende Vollständigkeitsprüfung des Datensatzes durchgeführt. Dabei wird zunächst für jede Variable sowie für den gesamten Datensatz überprüft, ob noch fehlende Werte vorhanden sind.

Zusätzlich wird kontrolliert, ob Beobachtungen mit mindestens einem fehlenden Wert verbleiben und ob die ursprüngliche Anzahl der Beobachtungen und Variablen erhalten geblieben ist.

Besondere Aufmerksamkeit gilt der Zielvariable Rezession. Für diese wird nochmals sichergestellt, dass keine fehlenden Werte vorliegen und die Anzahl der Rezessionsmonate unverändert geblieben ist.

In [44]:
print("--- Finale Kontrolle fehlender Werte ---")

# Fehlende Werte pro Spalte
missing_per_column = df.isna().sum()

# Nur Spalten mit fehlenden Werten
missing_columns = missing_per_column[missing_per_column > 0]

print("\nFehlende Werte pro Variable:")

if missing_columns.empty:
    print("Keine fehlenden Werte vorhanden.")
else:
    print(missing_columns)

# Gesamtzahl
print("\nGesamtzahl fehlender Werte:")
print(df.isna().sum().sum())

# Beobachtungen mit mindestens einem fehlenden Wert
print("\nBeobachtungen mit mindestens einem fehlenden Wert:")
print(df.isna().any(axis=1).sum())

# Zusätzliche Kontrolle
print("\nAnzahl Beobachtungen:", len(df))
print("Anzahl Variablen:", df.shape[1])

print("\nZielvariable Rezession:")
print("Fehlende Werte:", df["Rezession"].isna().sum())
print("Rezessionsmonate:", int(df["Rezession"].sum()))

--- Finale Kontrolle fehlender Werte ---

Fehlende Werte pro Variable:
Keine fehlenden Werte vorhanden.

Gesamtzahl fehlender Werte:
0

Beobachtungen mit mindestens einem fehlenden Wert:
0

Anzahl Beobachtungen: 425
Anzahl Variablen: 30

Zielvariable Rezession:
Fehlende Werte: 0
Rezessionsmonate: 88


**Prüfung auf Duplikate**

Nach der Behandlung der fehlenden Werte wird der Datensatz auf Duplikate überprüft. Dabei werden zwei mögliche Formen von Doppelungen betrachtet: vollständig identische Zeilen sowie mehrfach vorkommende Werte in der Zeitvariable Datum.

Da der Datensatz eine monatliche Zeitreihe darstellt, sollte jeder Monat genau einmal enthalten sein. Doppelte Datumswerte könnten auf Fehler bei der Zusammenführung oder Aufbereitung der verschiedenen Zeitreihen hinweisen.

In [45]:
print("--- Prüfung auf Duplikate ---")

# Vollständig identische Zeilen
duplicate_rows = df.duplicated().sum()

# Doppelte Zeitpunkte
duplicate_dates = df["Datum"].duplicated().sum()

print("Vollständig doppelte Zeilen:", duplicate_rows)
print("Doppelte Datumswerte:", duplicate_dates)

if duplicate_dates > 0:
    print("\nBetroffene Datumswerte:")
    print(
        df[df["Datum"].duplicated(keep=False)]
        .sort_values("Datum")
    )
else:
    print("\nJeder Monat kommt genau einmal vor.")

--- Prüfung auf Duplikate ---
Vollständig doppelte Zeilen: 0
Doppelte Datumswerte: 0

Jeder Monat kommt genau einmal vor.


**Transformation nicht-stationärer Reihen**

Vor der eigentlichen Transformation wird die Stationarität der numerischen Variablen untersucht. Hierfür werden der Augmented-Dickey-Fuller-Test (ADF) und der KPSS-Test verwendet. Beide Tests betrachten Stationarität aus unterschiedlichen Perspektiven und ermöglichen dadurch eine fundiertere Beurteilung der Zeitreiheneigenschaften.

Für jede numerische Variable werden Teststatistik, p-Wert und die jeweilige Entscheidung ermittelt. Anschließend werden die Ergebnisse beider Tests zu einer Gesamtentscheidung zusammengeführt. Die Resultate werden im DataFrame stationarity_df gespeichert.

Dieser Schritt dient ausschließlich der Diagnose und verändert die ursprünglichen Daten nicht. Die Ergebnisse bilden die Grundlage für die anschließende Entscheidung, welche nicht-stationären Variablen transformiert werden müssen

In [46]:
from statsmodels.tsa.stattools import adfuller, kpss
import pandas as pd
import numpy as np

print("\n--- Umfassende Stationaritätsanalyse (ADF- und KPSS-Tests) ---")

# Funktion zur Durchführung des Augmented Dickey-Fuller Tests
def perform_adfuller_test(series):
    result = adfuller(series)
    p_value = result[1]
    is_stationary = 'Stationär' if p_value <= 0.05 else 'Nicht stationär'
    return result[0], p_value, is_stationary

# Funktion zur Durchführung des KPSS-Tests
def perform_kpss_test(series):
    # KPSS Nullhypothese: Die Zeitreihe ist stationär um eine Trendlinie
    # result[0] ist die Teststatistik, result[1] der p-Wert
    try:
        result = kpss(series, regression='c', nlags='auto') # # KPSS regression='c': Test auf Level-Stationarität
# KPSS regression='ct': Test auf Trend-Stationarität
        p_value = result[1]
        # KPSS: p > 0.05 => stationär, p <= 0.05 => nicht stationär
        is_stationary = 'Stationär' if p_value > 0.05 else 'Nicht stationär'
        return result[0], p_value, is_stationary
    except ValueError as e:
        # Sometimes KPSS test encounters issues with too few data points or specific series characteristics
        return np.nan, np.nan, f'Fehler: {e}'

# Initialisiere eine Liste, um die Testergebnisse zu speichern
stationarity_results = []

# Führe Tests für alle numerischen Spalten durch
numerical_columns = df.select_dtypes(include=np.number).columns

for col in numerical_columns:
    # Sicherstellen, dass keine NaNs vorhanden sind (sollten nach Schritt 9 nicht mehr der Fall sein)
    series = df[col].dropna()

    if len(series) < 5: # Mindestanzahl für ADF
        stationarity_results.append({
            'Variable': col,
            'ADF-Teststatistik': np.nan,
            'ADF-p-Wert': np.nan,
            'ADF-Entscheidung': 'Nicht genügend Datenpunkte',
            'KPSS-Teststatistik': np.nan,
            'KPSS-p-Wert': np.nan,
            'KPSS-Entscheidung': 'Nicht genügend Datenpunkte',
            'Gesamtentscheidung': 'Nicht genügend Datenpunkte'
        })
        continue

    # ADF Test
    adf_stat, adf_p, adf_decision = perform_adfuller_test(series)

    # KPSS Test
    kpss_stat, kpss_p, kpss_decision = perform_kpss_test(series)

    # Kombinierte Entscheidung
    # ADF: p <= 0.05 -> Stationär
    # KPSS: p > 0.05 -> Stationär
    # Wenn beide "stationär" sagen, dann ist sie stationär
    # Wenn beide "nicht stationär" sagen, dann ist sie nicht stationär
    # Gemischte Ergebnisse erfordern Interpretation
    gesamt_entscheidung = ''
    if adf_decision == 'Stationär' and kpss_decision == 'Stationär':
        gesamt_entscheidung = 'Stationär (beide Tests)'
    elif adf_decision == 'Nicht stationär' and kpss_decision == 'Nicht stationär':
        gesamt_entscheidung = 'Nicht stationär (beide Tests)'
    elif adf_decision == 'Stationär' and kpss_decision == 'Nicht stationär':
        gesamt_entscheidung = 'Trend-Stationär (ADF stationär, KPSS nicht-level-stationär)'
    elif adf_decision == 'Nicht stationär' and kpss_decision == 'Stationär':
        gesamt_entscheidung = 'Differenzen-Stationär (ADF nicht-stationär, KPSS level-stationär)'
    else: # Fall für NaN oder Fehler im KPSS
        gesamt_entscheidung = f'Interpretation erforderlich (ADF: {adf_decision}, KPSS: {kpss_decision})'

    stationarity_results.append({
        'Variable': col,
        'ADF-Teststatistik': f'{adf_stat:.4f}',
        'ADF-p-Wert': f'{adf_p:.4f}',
        'ADF-Entscheidung': adf_decision,
        'KPSS-Teststatistik': f'{kpss_stat:.4f}' if not pd.isna(kpss_stat) else 'N/A',
        'KPSS-p-Wert': f'{kpss_p:.4f}' if not pd.isna(kpss_p) else 'N/A',
        'KPSS-Entscheidung': kpss_decision,
        'Gesamtentscheidung': gesamt_entscheidung
    })

# Erstelle DataFrame aus den Ergebnissen
stationarity_df = pd.DataFrame(stationarity_results)

print("\nErgebnistabelle der Stationaritätsanalyse:")
display(stationarity_df)

print("Umfassende Stationaritätsanalyse abgeschlossen.")


--- Umfassende Stationaritätsanalyse (ADF- und KPSS-Tests) ---


/tmp/ipykernel_2668/1634701336.py:19: InterpolationWarning: The test statistic is outside of the range of p-values available in the
look-up table. The actual p-value is smaller than the p-value returned.

  result = kpss(series, regression='c', nlags='auto') # # KPSS regression='c': Test auf Level-Stationarität
/tmp/ipykernel_2668/1634701336.py:19: InterpolationWarning: The test statistic is outside of the range of p-values available in the
look-up table. The actual p-value is smaller than the p-value returned.

  result = kpss(series, regression='c', nlags='auto') # # KPSS regression='c': Test auf Level-Stationarität
/tmp/ipykernel_2668/1634701336.py:19: InterpolationWarning: The test statistic is outside of the range of p-values available in the
look-up table. The actual p-value is smaller than the p-value returned.

  result = kpss(series, regression='c', nlags='auto') # # KPSS regression='c': Test auf Level-Stationarität
/tmp/ipykernel_2668/1634701336.py:19: InterpolationWarning: T


Ergebnistabelle der Stationaritätsanalyse:


/tmp/ipykernel_2668/1634701336.py:19: InterpolationWarning: The test statistic is outside of the range of p-values available in the
look-up table. The actual p-value is smaller than the p-value returned.

  result = kpss(series, regression='c', nlags='auto') # # KPSS regression='c': Test auf Level-Stationarität
/tmp/ipykernel_2668/1634701336.py:19: InterpolationWarning: The test statistic is outside of the range of p-values available in the
look-up table. The actual p-value is smaller than the p-value returned.

  result = kpss(series, regression='c', nlags='auto') # # KPSS regression='c': Test auf Level-Stationarität
/tmp/ipykernel_2668/1634701336.py:19: InterpolationWarning: The test statistic is outside of the range of p-values available in the
look-up table. The actual p-value is smaller than the p-value returned.

  result = kpss(series, regression='c', nlags='auto') # # KPSS regression='c': Test auf Level-Stationarität
/tmp/ipykernel_2668/1634701336.py:19: InterpolationWarning: T

,Variable,ADF-Teststatistik,ADF-p-Wert,ADF-Entscheidung,KPSS-Teststatistik,KPSS-p-Wert,KPSS-Entscheidung,Gesamtentscheidung
0,Industrial_Production,-1.6622,0.4507,Nicht stationär,2.4602,0.0100,Nicht stationär,Nicht stationär (beide Tests)
1,Manufacturing_Output,-1.6263,0.4694,Nicht stationär,2.5903,0.0100,Nicht stationär,Nicht stationär (beide Tests)
2,Capital_Goods_Output,-1.5166,0.5253,Nicht stationär,2.7606,0.0100,Nicht stationär,Nicht stationär (beide Tests)
3,Intermediate_Goods_Output,-1.6627,0.4504,Nicht stationär,2.4783,0.0100,Nicht stationär,Nicht stationär (beide Tests)
4,Consumer_Goods_Output,-2.2561,0.1865,Nicht stationär,0.7412,0.0100,Nicht stationär,Nicht stationär (beide Tests)
5,Orders_Abroad_Intermediate_Capital,-1.5756,0.4959,Nicht stationär,3.0521,0.0100,Nicht stationär,Nicht stationär (beide Tests)
6,Construction_Orders,-1.7537,0.4037,Nicht stationär,1.3538,0.0100,Nicht stationär,Nicht stationär (beide Tests)
7,Unemployment_Rate,-0.9340,0.7766,Nicht stationär,2.3673,0.0100,Nicht stationär,Nicht stationär (beide Tests)
8,Employment,-0.1177,0.9476,Nicht stationär,3.2219,0.0100,Nicht stationär,Nicht stationär (beide Tests)
9,CPI,1.4566,0.9974,Nicht stationär,3.1657,0.0100,Nicht stationär,Nicht stationär (beide Tests)


Umfassende Stationaritätsanalyse abgeschlossen.


**Transformation der Variablen**

Auf Grundlage der zuvor durchgeführten Stationaritätsanalyse werden die Variablen für die Machine-Learning-Modellierung transformiert. Die Transformation erfolgt auf einer Kopie des Datensatzes (df_transformed), sodass der ursprüngliche aufbereitete Datensatz unverändert erhalten bleibt.

Je nach Eigenschaften der jeweiligen Zeitreihe werden unterschiedliche Verfahren angewendet. Produktions-, Auftrags-, Beschäftigungs-, Außenhandels- und ausgewählte Marktvariablen werden mittels Log-Differenz
Δln(xt) transformiert. Für CPI, M1_Index und M3_Index wird die Year-over-Year-Wachstumsrate über zwölf Monate berechnet. Die Zinsvariablen German_Bond_Yield und Euribor_3M werden durch die erste Differenz transformiert. Bei weiteren Variablen, darunter Unemployment_Rate, Business_Situation_Services, Business_Situation_Retail und Term_Spread_Germany, wird die Transformation anhand der Ergebnisse der vorherigen ADF-/KPSS-Analyse entschieden. Ausgewählte Variablen werden im ursprünglichen Level belassen.

Ziel dieser Transformationen ist es, Trends und nicht-stationäre Strukturen zu reduzieren und die Zeitreihen für die anschließende Modellierung geeigneter zu machen.

In [47]:
import numpy as np
import pandas as pd

# Start with a copy of the original DataFrame to apply transformations
df_transformed = df.copy()

print("--- Finale Transformation der Variablen für Machine Learning ---")

# --- 1. Log-Differenz (DLN) ---
log_diff_variables = [
    'Industrial_Production', 'Manufacturing_Output', 'Capital_Goods_Output',
    'Intermediate_Goods_Output', 'Consumer_Goods_Output', 'Orders_Abroad_Intermediate_Capital',
    'Construction_Orders', 'Orders_Inflow', 'Domestic_Capital_Goods_Orders',
    'Domestic_Intermediate_Goods_Orders', 'Employment', 'Brent_Oil',
    'Exchange_Rate_USD_EUR', 'Exports', 'Imports'
]

print("\nAnwendung der Log-Differenz (DLN = np.log(x).diff()) für ausgewählte Variablen:")
for col in log_diff_variables:
    # Begründung: Die logarithmische Differenz (DLN) ist eine gängige Transformation für Variablen, die Wachstumsraten
    # abbilden oder deren Varianz mit dem Niveau zunimmt. Sie hilft, die Varianz zu stabilisieren und die Zeitreihe
    # stationärer zu machen, was für die Modellierung vorteilhaft ist.
    # Voraussetzung: Alle Werte müssen positiv sein, da der Logarithmus sonst undefiniert ist.
    if (df_transformed[col] <= 0).any():
        print(f"WARNUNG: Variable '{col}' enthält nicht-positive Werte. Die Log-Transformation wurde übersprungen, um Fehler zu vermeiden. Überprüfen Sie die Daten oder passen Sie die Strategie an.")
    else:
        df_transformed[col] = np.log(df_transformed[col]).diff()
        print(f"- '{col}' wurde mittels Log-Differenz transformiert.")

# --- 2. Year-over-Year-Wachstumsrate (YoY) ---
yoy_variables = [
    'CPI', 'M1_Index', 'M3_Index'
]

print("\nBerechnung der Year-over-Year-Wachstumsrate (YoY = pct_change(periods=12)) für ausgewählte Variablen:")
for col in yoy_variables:
    # Begründung: Die Jahreswachstumsrate (Year-over-Year, YoY) ist eine effektive Methode, um saisonale Muster zu eliminieren
    # und die zugrunde liegenden Trends und Zyklen in Zeitreihen, wie Preisindizes oder Geldmengen, zu isolieren. Sie macht
    # die Zeitreihe oft stationär und ist intuitiv als jährliche Veränderung interpretierbar.
    df_transformed[col] = df_transformed[col].pct_change(periods=12)
    print(f"- '{col}' wurde mittels YoY-Wachstumsrate transformiert.")

# --- 3. Erste Differenz ---
first_diff_explicit_variables = [
    'German_Bond_Yield', 'Euribor_3M'
]

print("\nAnwendung der ersten Differenz (diff()) für ausgewählte Variablen:")
for col in first_diff_explicit_variables:
    # Begründung: Die erste Differenz entfernt lineare Trends aus einer Zeitreihe und macht sie oft stationär.
    # Dies ist eine Standardpraxis für Zinsraten, die häufig einem Zufallsgang-Prozess ähneln.
    df_transformed[col] = df_transformed[col].diff()
    print(f"- '{col}' wurde mittels erster Differenz transformiert.")

# Erste Differenz für 'Unemployment_Rate' basierend auf Stationaritätstests
# 'stationarity_df' aus vorherigem Schritt ist im Kernel verfügbar.
unemployment_rate_stationary_decision = stationarity_df[stationarity_df['Variable'] == 'Unemployment_Rate']['Gesamtentscheidung'].iloc[0]
if unemployment_rate_stationary_decision == 'Nicht stationär (beide Tests)':
    # Begründung: Die Arbeitslosenquote wurde in der vorherigen Analyse als nicht stationär identifiziert.
    # Die erste Differenzierung hilft, den Trend zu entfernen und Stationarität zu erreichen, was für stabilere Modellierung und bessere Vergleichbarkeit hilfreich ist
    df_transformed['Unemployment_Rate'] = df_transformed['Unemployment_Rate'].diff()
    print(f"- 'Unemployment_Rate' wurde mittels erster Differenz transformiert (basierend auf Stationaritätstests: {unemployment_rate_stationary_decision}).")
else:
    # Begründung: Die Arbeitslosenquote wurde als stationär befunden oder erfordert keine Differenzierung nach den Tests.
    print(f"- 'Unemployment_Rate' verbleibt im Level (basierend auf Stationaritätstests: {unemployment_rate_stationary_decision}).")


# --- 4. Im Level belassen oder Differenzieren basierend auf Stationaritätstests ---
print("\nVariablen, die im Level belassen oder basierend auf Stationaritätstests differenziert werden:")

# Variablen, die explizit im Level bleiben sollen (unabhängig von Stationaritätstests)
level_variables_explicit = [
    'Inflation_Rate', 'Consumer_Confidence', 'Production_Expectations_Manufacturing'
]

for col in level_variables_explicit:
    # Begründung: Diese Variablen sollen gemäß Aufgabenstellung explizit im Level belassen werden, auch wenn
    # die Stationaritätstests möglicherweise auf Nicht-Stationarität hindeuten. Sie werden oft in ihrer Originalform
    # als Indikatoren verwendet und interpretiert.
    print(f"- '{col}' verbleibt explizit im Level (gemäß Aufgabenstellung).")
    # Keine Aktion erforderlich, da sie bereits in df_transformed in ihrer ursprünglichen Form vorliegen.

# Variablen, für die die Entscheidung basierend auf Stationaritätstests getroffen wird
decision_based_variables = [
    'Effective_Exchange_Rate', 'Business_Situation_Services',
    'Business_Situation_Retail', 'Term_Spread_Germany'
]

for col in decision_based_variables:
    stationary_decision = stationarity_df[stationarity_df['Variable'] == col]['Gesamtentscheidung'].iloc[0]
    if stationary_decision == 'Nicht stationär (beide Tests)':
        # Begründung: Diese Variable wurde in der vorherigen Analyse als nicht stationär identifiziert.
        # Die erste Differenzierung hilft, den Trend zu entfernen und Stationarität zu erreichen.
        df_transformed[col] = df_transformed[col].diff()
        print(f"- '{col}' wurde mittels erster Differenz transformiert (basierend auf Stationaritätstests: {stationary_decision}).")
    else:
        # Begründung: Diese Variable wurde als stationär befunden oder erfordert keine Differenzierung nach den Tests.
        print(f"- '{col}' verbleibt im Level (basierend auf Stationaritätstests: {stationary_decision}).")
        # Keine Aktion erforderlich.

# --- Finaler Bereinigungsschritt ---
print("\nBereinigung des transformierten DataFrames: Entfernen aller Zeilen mit NaN-Werten, die durch die Transformationen entstanden sind.")
initial_rows = df_transformed.shape[0]
df_transformed = df_transformed.dropna().reset_index(drop=True)
rows_after_dropna = df_transformed.shape[0]
print("Finaler Zeitraum:", df_transformed["Datum"].min(), "bis", df_transformed["Datum"].max())

transformation_summary = pd.DataFrame({
    "Variable": [
        "Industrial_Production",
        "Manufacturing_Output",
        "Capital_Goods_Output",
        "Intermediate_Goods_Output",
        "Consumer_Goods_Output",
        "Orders_Abroad_Intermediate_Capital",
        "Construction_Orders",
        "Orders_Inflow",
        "Domestic_Capital_Goods_Orders",
        "Domestic_Intermediate_Goods_Orders",
        "Employment",
        "Brent_Oil",
        "Exchange_Rate_USD_EUR",
        "Exports",
        "Imports",
        "CPI",
        "M1_Index",
        "M3_Index",
        "German_Bond_Yield",
        "Euribor_3M",
        "Unemployment_Rate",
        "Inflation_Rate",
        "Consumer_Confidence",
        "Production_Expectations_Manufacturing",
        "Effective_Exchange_Rate",
        "Business_Situation_Services",
        "Business_Situation_Retail",
        "Term_Spread_Germany"
    ],

    "Transformation": [
        "Log-Differenz (DLN)",
        "Log-Differenz (DLN)",
        "Log-Differenz (DLN)",
        "Log-Differenz (DLN)",
        "Log-Differenz (DLN)",
        "Log-Differenz (DLN)",
        "Log-Differenz (DLN)",
        "Log-Differenz (DLN)",
        "Log-Differenz (DLN)",
        "Log-Differenz (DLN)",
        "Log-Differenz (DLN)",
        "Log-Differenz (DLN)",
        "Log-Differenz (DLN)",
        "Log-Differenz (DLN)",
        "Log-Differenz (DLN)",

        "YoY-Wachstumsrate",
        "YoY-Wachstumsrate",
        "YoY-Wachstumsrate",

        "Erste Differenz",
        "Erste Differenz",

        "Erste Differenz",      # Unemployment_Rate

        "Level",                # Inflation_Rate
        "Level",                # Consumer_Confidence
        "Level",                # Production_Expectations_Manufacturing

        "Level",                # Effective_Exchange_Rate
        "Erste Differenz",      # Business_Situation_Services
        "Erste Differenz",      # Business_Situation_Retail
        "Erste Differenz"       # Term_Spread_Germany
    ]
})

print("\nZusammenfassung der Transformationen:")
display(transformation_summary)

# --- Ausgabe der Ergebnisse ---
print(f"\nGröße des transformierten DataFrames (Zeilen, Spalten): {df_transformed.shape}")

print("\nErste 5 Beobachtungen des transformierten DataFrames:")
display(df_transformed.head())

print("\nVerbleibende fehlende Werte nach Transformation und dropna:")
display(df_transformed.isnull().sum())

df_transformed.to_excel(
    "Final_Dataset_ML_1991_2026.xlsx",
    index=False
)

transformation_summary.to_excel(
    "Transformation_Summary.xlsx",
    index=False
)

print("\n Finaler Datensatz gespeichert.")
print("Transformation Summary gespeichert.")

--- Finale Transformation der Variablen für Machine Learning ---

Anwendung der Log-Differenz (DLN = np.log(x).diff()) für ausgewählte Variablen:
- 'Industrial_Production' wurde mittels Log-Differenz transformiert.
- 'Manufacturing_Output' wurde mittels Log-Differenz transformiert.
- 'Capital_Goods_Output' wurde mittels Log-Differenz transformiert.
- 'Intermediate_Goods_Output' wurde mittels Log-Differenz transformiert.
- 'Consumer_Goods_Output' wurde mittels Log-Differenz transformiert.
- 'Orders_Abroad_Intermediate_Capital' wurde mittels Log-Differenz transformiert.
- 'Construction_Orders' wurde mittels Log-Differenz transformiert.
- 'Orders_Inflow' wurde mittels Log-Differenz transformiert.
- 'Domestic_Capital_Goods_Orders' wurde mittels Log-Differenz transformiert.
- 'Domestic_Intermediate_Goods_Orders' wurde mittels Log-Differenz transformiert.
- 'Employment' wurde mittels Log-Differenz transformiert.
- 'Brent_Oil' wurde mittels Log-Differenz transformiert.
- 'Exchange_Rate_USD_EU

,Variable,Transformation
0,Industrial_Production,Log-Differenz (DLN)
1,Manufacturing_Output,Log-Differenz (DLN)
2,Capital_Goods_Output,Log-Differenz (DLN)
3,Intermediate_Goods_Output,Log-Differenz (DLN)
4,Consumer_Goods_Output,Log-Differenz (DLN)
5,Orders_Abroad_Intermediate_Capital,Log-Differenz (DLN)
6,Construction_Orders,Log-Differenz (DLN)
7,Orders_Inflow,Log-Differenz (DLN)
8,Domestic_Capital_Goods_Orders,Log-Differenz (DLN)
9,Domestic_Intermediate_Goods_Orders,Log-Differenz (DLN)



Größe des transformierten DataFrames (Zeilen, Spalten): (413, 30)

Erste 5 Beobachtungen des transformierten DataFrames:


,Datum,Industrial_Production,Manufacturing_Output,Capital_Goods_Output,Intermediate_Goods_Output,Consumer_Goods_Output,Orders_Abroad_Intermediate_Capital,Construction_Orders,Unemployment_Rate,Employment,...,Domestic_Intermediate_Goods_Orders,Euribor_3M,Brent_Oil,Production_Expectations_Manufacturing,Business_Situation_Services,Business_Situation_Retail,Term_Spread_Germany,Exports,Imports,Rezession
0,1992-01-31,0.016159,0.016487,0.019311,0.006873,0.018915,-0.081346,0.054115,0.3,-0.000621,...,0.008299,-0.05,-0.013673,2.1,-0.938580,0.0,-0.2150,-0.028073,0.047531,0
1,1992-02-29,0.009816,0.012500,0.016261,0.010899,0.001970,0.008439,0.074848,-0.2,-0.000777,...,-0.021722,0.08,-0.006076,0.4,6.598765,-6.2,-0.1075,0.008749,-0.010206,0
2,1992-03-31,-0.023472,-0.028988,-0.036965,-0.020535,-0.023906,0.022162,-0.031749,0.0,-0.001193,...,0.018411,0.09,-0.023544,-0.6,-7.300699,-11.3,0.0175,0.015558,-0.012720,1
3,1992-04-30,-0.001251,-0.001280,0.004175,-0.006940,0.010030,-0.050573,-0.020051,0.1,-0.000805,...,-0.038889,0.05,0.070618,-1.7,-4.267983,0.0,-0.0425,0.021046,0.042676,1
4,1992-05-31,-0.011328,-0.009003,-0.008368,-0.001394,-0.017112,-0.020379,-0.020461,0.1,-0.000728,...,-0.001726,0.04,0.049998,-1.7,-2.872531,-3.7,0.0000,-0.094706,-0.061970,1



Verbleibende fehlende Werte nach Transformation und dropna:


,0
Datum,0
Industrial_Production,0
Manufacturing_Output,0
Capital_Goods_Output,0
Intermediate_Goods_Output,0
Consumer_Goods_Output,0
Orders_Abroad_Intermediate_Capital,0
Construction_Orders,0
Unemployment_Rate,0
Employment,0



 Finaler Datensatz gespeichert.
Transformation Summary gespeichert.


**Zusammenfassung der Transformation**

In [48]:
summary_table = (
    transformation_summary
    .groupby("Transformation")
    .agg(
        Anzahl=("Variable", "count"),
        Beispiele=("Variable", lambda x: ", ".join(x.head(3)))
    )
    .reset_index()
)

# Schönere Namen
summary_table["Transformation"] = summary_table["Transformation"].replace({
    "Log-Differenz (DLN)": "Log-Differenz (DLN)",
    "YoY-Wachstumsrate": "Year-over-Year (YoY)",
    "Erste Differenz": "Erste Differenz",
    "Level": "Level"
})

print("Zusammenfassung der Transformationen:")
display(summary_table)

# Optional als Excel speichern
summary_table.to_excel("Transformation_Summary_Short.xlsx", index=False)

Zusammenfassung der Transformationen:


,Transformation,Anzahl,Beispiele
0,Erste Differenz,6,"German_Bond_Yield, Euribor_3M, Unemployment_Rate"
1,Level,4,"Inflation_Rate, Consumer_Confidence, Productio..."
2,Log-Differenz (DLN),15,"Industrial_Production, Manufacturing_Output, C..."
3,Year-over-Year (YoY),3,"CPI, M1_Index, M3_Index"


**Erneute ADF-/KPSS-Prüfung**

Nach der Transformation der makroökonomischen Variablen wird die Stationarität erneut mithilfe des ADF- und KPSS-Tests überprüft. Die Analyse wird diesmal auf dem transformierten Datensatz df_transformed durchgeführt.

Ziel dieser zweiten Prüfung ist es festzustellen, ob die angewandten Transformationen – Log-Differenz, Year-over-Year-Wachstumsrate und erste Differenz – die zuvor vorhandene Nicht-Stationarität erfolgreich reduziert bzw. beseitigt haben.

Die Ergebnisse beider Tests werden erneut zu einer Gesamtentscheidung zusammengeführt und in stationarity_df_transformed gespeichert. Dadurch kann überprüft werden, welche Variablen nach der Transformation als stationär gelten und bei welchen Variablen weiterhin eine genauere Betrachtung erforderlich ist.

In [49]:
from statsmodels.tsa.stattools import adfuller, kpss
import pandas as pd
import numpy as np

print("\n--- Erneute Stationaritätsanalyse (ADF- und KPSS-Tests) auf df_transformed ---")

# Funktion zur Durchführung des Augmented Dickey-Fuller Tests (bereits definiert, hier wiederholt zur Klarheit)
def perform_adfuller_test(series):
    result = adfuller(series)
    p_value = result[1]
    is_stationary = 'Stationär' if p_value <= 0.05 else 'Nicht stationär'
    return result[0], p_value, is_stationary

# Funktion zur Durchführung des KPSS-Tests (bereits definiert, hier wiederholt zur Klarheit)
def perform_kpss_test(series):
    try:
        result = kpss(series, regression='c', nlags='auto')
        p_value = result[1]
        is_stationary = 'Stationär' if p_value > 0.05 else 'Nicht stationär'
        return result[0], p_value, is_stationary
    except ValueError as e:
        return np.nan, np.nan, f'Fehler: {e}'

# Initialisiere eine Liste, um die Testergebnisse zu speichern
stationarity_results_transformed = []

# Führe Tests für alle numerischen Spalten in df_transformed durch
numerical_columns_transformed = df_transformed.select_dtypes(include=np.number).columns

for col in numerical_columns_transformed:
    # Sicherstellen, dass keine NaNs vorhanden sind (sollten nach dropna nicht mehr der Fall sein)
    series = df_transformed[col].dropna()

    if len(series) < 5: # Mindestanzahl für ADF
        stationarity_results_transformed.append({
            'Variable': col,
            'ADF-Teststatistik': np.nan,
            'ADF-p-Wert': np.nan,
            'ADF-Entscheidung': 'Nicht genügend Datenpunkte',
            'KPSS-Teststatistik': np.nan,
            'KPSS-p-Wert': np.nan,
            'KPSS-Entscheidung': 'Nicht genügend Datenpunkte',
            'Gesamtentscheidung': 'Nicht genügend Datenpunkte'
        })
        continue

    # ADF Test
    adf_stat, adf_p, adf_decision = perform_adfuller_test(series)

    # KPSS Test
    kpss_stat, kpss_p, kpss_decision = perform_kpss_test(series)

    # Kombinierte Entscheidung
    gesamt_entscheidung = ''
    if adf_decision == 'Stationär' and kpss_decision == 'Stationär':
        gesamt_entscheidung = 'Stationär (beide Tests)'
    elif adf_decision == 'Nicht stationär' and kpss_decision == 'Nicht stationär':
        gesamt_entscheidung = 'Nicht stationär (beide Tests)'
    elif adf_decision == 'Stationär' and kpss_decision == 'Nicht stationär':
        gesamt_entscheidung = 'Trend-Stationär (ADF stationär, KPSS nicht-level-stationär)'
    elif adf_decision == 'Nicht stationär' and kpss_decision == 'Stationär':
        gesamt_entscheidung = 'Differenzen-Stationär (ADF nicht-stationär, KPSS level-stationär)'
    else: # Fall für NaN oder Fehler im KPSS
        gesamt_entscheidung = f'Interpretation erforderlich (ADF: {adf_decision}, KPSS: {kpss_decision})'

    stationarity_results_transformed.append({
        'Variable': col,
        'ADF-Teststatistik': f'{adf_stat:.4f}',
        'ADF-p-Wert': f'{adf_p:.4f}',
        'ADF-Entscheidung': adf_decision,
        'KPSS-Teststatistik': f'{kpss_stat:.4f}' if not pd.isna(kpss_stat) else 'N/A',
        'KPSS-p-Wert': f'{kpss_p:.4f}' if not pd.isna(kpss_p) else 'N/A',
        'KPSS-Entscheidung': kpss_decision,
        'Gesamtentscheidung': gesamt_entscheidung
    })

# Erstelle DataFrame aus den Ergebnissen
stationarity_df_transformed = pd.DataFrame(stationarity_results_transformed)

print("\nErgebnistabelle der Stationaritätsanalyse für df_transformed:")
display(stationarity_df_transformed)

print("Erneute Stationaritätsanalyse abgeschlossen.")


--- Erneute Stationaritätsanalyse (ADF- und KPSS-Tests) auf df_transformed ---


/tmp/ipykernel_2668/1449917797.py:17: InterpolationWarning: The test statistic is outside of the range of p-values available in the
look-up table. The actual p-value is greater than the p-value returned.

  result = kpss(series, regression='c', nlags='auto')
/tmp/ipykernel_2668/1449917797.py:17: InterpolationWarning: The test statistic is outside of the range of p-values available in the
look-up table. The actual p-value is greater than the p-value returned.

  result = kpss(series, regression='c', nlags='auto')
/tmp/ipykernel_2668/1449917797.py:17: InterpolationWarning: The test statistic is outside of the range of p-values available in the
look-up table. The actual p-value is greater than the p-value returned.

  result = kpss(series, regression='c', nlags='auto')
/tmp/ipykernel_2668/1449917797.py:17: InterpolationWarning: The test statistic is outside of the range of p-values available in the
look-up table. The actual p-value is greater than the p-value returned.

  result = kpss(se


Ergebnistabelle der Stationaritätsanalyse für df_transformed:


/tmp/ipykernel_2668/1449917797.py:17: InterpolationWarning: The test statistic is outside of the range of p-values available in the
look-up table. The actual p-value is greater than the p-value returned.

  result = kpss(series, regression='c', nlags='auto')
/tmp/ipykernel_2668/1449917797.py:17: InterpolationWarning: The test statistic is outside of the range of p-values available in the
look-up table. The actual p-value is greater than the p-value returned.

  result = kpss(series, regression='c', nlags='auto')
/tmp/ipykernel_2668/1449917797.py:17: InterpolationWarning: The test statistic is outside of the range of p-values available in the
look-up table. The actual p-value is greater than the p-value returned.

  result = kpss(series, regression='c', nlags='auto')
/tmp/ipykernel_2668/1449917797.py:17: InterpolationWarning: The test statistic is outside of the range of p-values available in the
look-up table. The actual p-value is greater than the p-value returned.

  result = kpss(se

,Variable,ADF-Teststatistik,ADF-p-Wert,ADF-Entscheidung,KPSS-Teststatistik,KPSS-p-Wert,KPSS-Entscheidung,Gesamtentscheidung
0,Industrial_Production,-17.1768,0.0000,Stationär,0.1059,0.1000,Stationär,Stationär (beide Tests)
1,Manufacturing_Output,-17.1040,0.0000,Stationär,0.0962,0.1000,Stationär,Stationär (beide Tests)
2,Capital_Goods_Output,-15.1649,0.0000,Stationär,0.1055,0.1000,Stationär,Stationär (beide Tests)
3,Intermediate_Goods_Output,-19.4936,0.0000,Stationär,0.1105,0.1000,Stationär,Stationär (beide Tests)
4,Consumer_Goods_Output,-12.2729,0.0000,Stationär,0.0741,0.1000,Stationär,Stationär (beide Tests)
5,Orders_Abroad_Intermediate_Capital,-17.0703,0.0000,Stationär,0.0623,0.1000,Stationär,Stationär (beide Tests)
6,Construction_Orders,-21.4953,0.0000,Stationär,0.1658,0.1000,Stationär,Stationär (beide Tests)
7,Unemployment_Rate,-8.6819,0.0000,Stationär,0.2847,0.1000,Stationär,Stationär (beide Tests)
8,Employment,-4.8466,0.0000,Stationär,0.2931,0.1000,Stationär,Stationär (beide Tests)
9,CPI,-3.6366,0.0051,Stationär,0.2855,0.1000,Stationär,Stationär (beide Tests)


Erneute Stationaritätsanalyse abgeschlossen.


**Zielvariable definieren und überprüfen**

Vor der Modellierung wird Rezession explizit als Zielvariable definiert und abschließend überprüft. Dabei werden das Vorhandensein der Variable, ihre Klassenverteilung, der Datentyp sowie mögliche fehlende Werte kontrolliert. Die Zielvariable selbst wird durch die vorherigen Transformationen nicht verändert.

In [50]:
# Define the target variable
target_column_name = 'Rezession'

# Verify the target column again (redundant but good for explicit workflow steps)
if target_column_name in df.columns:
    print(f"Target variable '{target_column_name}' successfully identified.")
    print(f"Value counts for '{target_column_name}':\n{df[target_column_name].value_counts()}")
    print(f"Data type of '{target_column_name}': {df[target_column_name].dtype}")
else:
    print(f"Error: Target variable '{target_column_name}' not found in the DataFrame. Please check the column name.")

Target variable 'Rezession' successfully identified.
Value counts for 'Rezession':
Rezession
0    337
1     88
Name: count, dtype: int64
Data type of 'Rezession': int64


**Trennung von Datum, Zielvariable und Prädiktoren**

Nach Abschluss der Transformation und Stationaritätsprüfung wird der Datensatz für die spätere Machine-Learning-Modellierung strukturiert. Die Zeitvariable Datum wird separat gespeichert, Rezession als Zielvariable y definiert und alle verbleibenden makroökonomischen Variablen bilden die Prädiktormatrix X.

Dabei wird kontrolliert, dass weder Datum noch Rezession Bestandteil der Prädiktoren sind. Dadurch wird insbesondere verhindert, dass die Zielvariable versehentlich als Eingangsvariable des Modells verwendet wird.

In [51]:
# Separate Date, Target, and Predictor variables

# Date column
date_column = df['Datum']
print(f"'Datum' column successfully extracted (first 5 values):\n{date_column.head()}\n")

# Target variable
y = df[target_column_name]
print(f"Target variable '{target_column_name}' successfully extracted (first 5 values):\n{y.head()}\n")

# Predictor variables (all columns except 'Datum' and the target variable)
X = df.drop(columns=['Datum', target_column_name])
print(f"Predictor variables (X) successfully extracted. Shape: {X.shape}")
print(f"First 5 rows of Predictor variables:\n{X.head()}\n")

# Verify no date column in X
if 'Datum' in X.columns:
    print("Error: 'Datum' column found in predictor variables X. It should have been dropped.")
else:
    print("'Datum' column successfully excluded from predictor variables X.")

# Verify target column is not in X
if target_column_name in X.columns:
    print(f"Error: Target variable '{target_column_name}' found in predictor variables X. It should have been dropped.")
else:
    print(f"Target variable '{target_column_name}' successfully excluded from predictor variables X.")

'Datum' column successfully extracted (first 5 values):
0   1991-01-31
1   1991-02-28
2   1991-03-31
3   1991-04-30
4   1991-05-31
Name: Datum, dtype: datetime64[ns]

Target variable 'Rezession' successfully extracted (first 5 values):
0    0
1    0
2    0
3    0
4    0
Name: Rezession, dtype: int64

Predictor variables (X) successfully extracted. Shape: (425, 28)
First 5 rows of Predictor variables:
   Industrial_Production  Manufacturing_Output  Capital_Goods_Output  \
0                   81.8                  80.4                  74.0   
1                   80.9                  79.2                  72.6   
2                   80.2                  78.5                  72.3   
3                   79.5                  77.9                  71.3   
4                   78.7                  76.9                  70.2   

   Intermediate_Goods_Output  Consumer_Goods_Output  \
0                       72.5                  105.0   
1                       71.5                  103.1  

In [52]:
import os

output_dir = "/content/drive/MyDrive/Masterarbeit/Data_processing"

# Ordner sicherstellen
os.makedirs(output_dir, exist_ok=True)

output_file = os.path.join(
    output_dir,
    "Masterarbeit_Dataset_ML_1992_2026_Preprocessed.xlsx"
)

# Transformierten Datensatz speichern
df_transformed.to_excel(output_file, index=False)

print("Finaler ML-Datensatz erfolgreich gespeichert:")
print(output_file)

print("\nDimension:", df_transformed.shape)
print("Zeitraum:", df_transformed["Datum"].min(), "bis", df_transformed["Datum"].max())
print("Fehlende Werte:", df_transformed.isna().sum().sum())
print("Rezessionsmonate:", int(df_transformed["Rezession"].sum()))

Finaler ML-Datensatz erfolgreich gespeichert:
/content/drive/MyDrive/Masterarbeit/Data_processing/Masterarbeit_Dataset_ML_1992_2026_Preprocessed.xlsx

Dimension: (413, 30)
Zeitraum: 1992-01-31 00:00:00 bis 2026-05-31 00:00:00
Fehlende Werte: 0
Rezessionsmonate: 88
